# Week 8: Web Scraping Part 2
**Name**: Isabelle Sakelaris \
**Date**: October 19, 2026

In [5]:
import requests

url = "https://www.gutenberg.org/cache/epub/22403/pg22403-images.html"
headers = {"User-Agent": "INFO664-class-project (educational)"}

response = requests.get(url, headers=headers)

with open("gutenberg.html", "w", encoding="utf-8") as file:
    file.write(response.text)

In [12]:
import csv
import json # allows us to write json files and interpret requests as JSON
import os # operating system library, let's us do things like making new folders
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

In [13]:
SCRAPED_PAGES_DIRECTORY = "scraped_pages/"
SCRAPED_PAGES_PREFIX = "page"
scraped_pages_count = 0 # keep track of how many pages we have scraped

headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    global scraped_pages_count  # use the scraped_pages_count from the top of the script, instead of making a new local one
    filename = f"{SCRAPED_PAGES_DIRECTORY}{SCRAPED_PAGES_PREFIX}{scraped_pages_count}.html"
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        scraped_pages_count = scraped_pages_count + 1
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)
        scraped_pages_count = scraped_pages_count + 1
    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

In [19]:
url = "https://www.gutenberg.org/cache/epub/22403/pg22403-images.html"
html = fetch_page(url) # filling the var "html" with the actual html, as text
soup = BeautifulSoup(html, "html.parser") # new var, converting the text into something structured using the Beautiful Soup library

Now we will look at the data from the website!

In [81]:
title_element = soup.find_all(name="i")
for index, child in enumerate(title_element):
    if child.get_text(strip=True)[0].isdigit(): # zero index is to get the number
        x = child.get_text(strip=True)[0]
        poem_title = child.get_text(strip=True).replace(x, "")
        # print(child.get_text(strip=True))
        print(poem_title)
# next step could be to remove the number and find the text that follows.

6
6
For a Picture ofSt. Dorothea
Heaven—HavenA nun takes the veil

The Habit of Perfection
THE WRECKOF THE DEUTSCHLAND

Penmaen Pool
The Silver Jubilee:To James First Bishop of Shrewsbury on the 25th Yearof his Episcopate July 28. 187

God's Grandeur

The Starlight Night

Spring
0
The Lantern out of Doors

The Sea and the Skylark
3
Pied Beauty
4
Hurrahing in Harvest
5
Caged Skylark
6
In the Valley of the Elwy
8
The May Magnificat
0
Duns Scotus's Oxford
1
Henry Purcell

Peace
4
Morning Midday and Evening Sacrifice
5
Andromeda
6
The Candle Indoors
9
Felix Randal
0
Brothers
2
Spelt from Sibyl's Leaves

Inversnaid
5
Ribblesdale
7The Blessed Virgin compared to the Air weBreathe
8
To what serves Mortal Beauty?
9
(The Soldier)
0
(Carrion Comfort)
1
3
Harry Ploughman

5
6
7
8
That Nature is a Heraclitean Fire and of
the comfort of the Resurrection
9In honour ofSt. Alphonsus RodriguezLaybrother of the Society of Jesus
0
1
To R. B.
2
Summa
3

6
(Ash-boughs)
7
9
0
1
Cheery Beggar
2
3
5
Moonrise



In [66]:
# use beautifulsoup next sibling to find the next elements to figure out if they're italicized or not
# look up "while loop" will keep repeating something as long as a condition is true.
    # while the next sibling is not an italicized element that starts with a number, pull this into the entry for the poem (to get a structured dataset for poem title and text)

### Going back to a class example of how to create a dataset from scraped data

In [85]:
entry_titles = [] 
entry_paragraphs = [] #put something in here

for child in title_element:
    entry_titles.append(child.get_text())

entry_dictionary = {
    "title": entry_titles,
    "paragraphs": entry_paragraphs
}

In [88]:
# change the logic below for your own data...

#def scrape_data(html, url):
    # soup = BeautifulSoup(html, "html.parser") #convert the html input into a soup
    # title_element = soup.find(class_="entry-title")
    # entry_title - title_element.get_text() # creating a var containing the text of the title
    # entry_content_element - soup.find(class_="entry-content") # creating a var for every paragraph in the entry content
    # entry_paragraphs = []
    # for child in children:
        # entry_paragraphs.append(child.get_text()) # append the para to the list of "entry_paragraphs"

    # entry_dictionary = {
        # "title": entry_title,
        # "paragraphs": entry_paragraphs,
        # "url": url
       # }
    # return entry_dictionary
# print(scrape_data(html, url)) # this will confirm that the function is outputting a dictionary w the scraped data

In [ ]:
#urls = [
   # "[a blog entry]",
   # "[a different blog entry]",
    #"[a third blog entry]"
# ]

# scraped_data = [] # will be an empty list for the dictionaries representing the scraped websites

# for u in urls: # for each of the urls in the list of urls...
    # html = fetch_page(u) # fetch the page, getting it from the web and loading it as an html string
    # website_data = scrape_data(html, u) # scrape the data in that string
    # scrape_data.append(website_data) #append that scraped data to the scraped_data list

# print(scraped_data)
# print(len(scraped_data)) # will tell you how many entries are in the dict!

To save a list of dictionaries as a json file we can load in other contexts, we use json.dump function!

In [ ]:
# with open("scraped_data.json", "w", encoding="utf-8") as file: #create a new file and refer to it as "file"
   # json.dump(scraped_data, file, indent=4, ensure_ascii=False) # ensure_ascii = false makes sure that you don't strip the data of "non-conforming" characters 